# 01 – Data understanding and EDA
Run `python -m src.download_data` first. After each figure, write 2–3 sentences of **your own** observations (what you see, why it matters, a limitation).

In [3]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from src import config as C
sns.set_theme()
df = pd.read_csv(C.RAW_FILE)
df.head()

Matplotlib is building the font cache; this may take a moment.


,UID,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Structure and data quality

In [4]:
print('Shape:', df.shape)
df.info()
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
df[C.NUMERIC].describe().T

Shape: (10000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UID                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), 

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.0,108.0,162.0,253.0


## 2. Target distribution

In [5]:
counts = df[C.TARGET].value_counts()
print(counts); print((counts/len(df)*100).round(2))
sns.countplot(x=C.TARGET, data=df); plt.title('Machine failure (0/1)'); plt.show()

Machine failure
0    9661
1     339
Name: count, dtype: int64
Machine failure
0    96.61
1     3.39
Name: count, dtype: float64


C:\Users\ABC\AppData\Local\Temp\ipykernel_15796\905499567.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  sns.countplot(x=C.TARGET, data=df); plt.title('Machine failure (0/1)'); plt.show()


**Observation (write yours):** 

## 3. Distributions

In [6]:
df[C.NUMERIC].hist(bins=30, figsize=(12,7)); plt.tight_layout(); plt.show()
sns.countplot(x='Type', data=df); plt.show()
print(df.groupby('Type')[C.TARGET].mean())

Type
H    0.020937
L    0.039167
M    0.027694
Name: Machine failure, dtype: float64


C:\Users\ABC\AppData\Local\Temp\ipykernel_15796\3169168763.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  df[C.NUMERIC].hist(bins=30, figsize=(12,7)); plt.tight_layout(); plt.show()
C:\Users\ABC\AppData\Local\Temp\ipykernel_15796\3169168763.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  sns.countplot(x='Type', data=df); plt.show()


**Observation (write yours):** 

## 4. Sensor values by failure label

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(14,7))
for ax, col in zip(axes.ravel(), C.NUMERIC):
    sns.boxplot(x=C.TARGET, y=col, data=df, ax=ax)
plt.tight_layout(); plt.show()

C:\Users\ABC\AppData\Local\Temp\ipykernel_15796\3252454607.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Observation (write yours):** 

## 5. Correlation (numeric predictors)

In [8]:
sns.heatmap(df[C.NUMERIC].corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.show()

C:\Users\ABC\AppData\Local\Temp\ipykernel_15796\3380852059.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  sns.heatmap(df[C.NUMERIC].corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.show()


**Observation (write yours):** correlation is not causation; note any strongly correlated pair (e.g. the two temperatures).

## 6. Leakage check
Confirm that TWF/HDF/PWF/OSF/RNF relate directly to the target, so they are excluded.

In [9]:
if set(C.LEAKAGE_COLUMNS) <= set(df.columns):
    print(df.groupby(C.TARGET)[C.LEAKAGE_COLUMNS].sum())
    print('Failures with no failure-mode flag:', int(((df[C.TARGET]==1)&(df[C.LEAKAGE_COLUMNS].sum(axis=1)==0)).sum()))

                 TWF  HDF  PWF  OSF  RNF
Machine failure                         
0                  0    0    0    0   18
1                 46  115   95   98    1
Failures with no failure-mode flag: 9
